# TIMIT HMM Batched GPU (Optimized)
This notebook contains the optimized code for batched HMM training on the TIMIT dataset.
Run cells sequentially. Requirements: `torch`, `torchaudio`, `scikit-learn`, `matplotlib`, `tqdm`.


In [ ]:

"""
TIMIT HMM (Discrete) — Batched GPU implementation (Optimized)
Includes: MFCC extraction, MiniBatch KMeans discretization, train/validation split,
Batched log-space Forward, Viterbi, Batched Baum–Welch (EM) with fully vectorized B_num,
training loop with validation likelihood logging, plotting, checkpointing, and
conversion-ready layout for a Jupyter notebook (.ipynb).

Requirements:
- Python 3.8+
- torch, torchaudio
- scikit-learn
- matplotlib, tqdm

Notes:
- This notebook is optimized to run on GPU using PyTorch. It uses log-space DP for
  numerical stability and vectorized accumulation for expected emission counts.
- Adjust SAMPLE_SUBSET and N_CLUSTERS to fit your GPU memory.
"""

import os
import math
import random
import json
from typing import List, Tuple

import torch
import torchaudio
from torchaudio.datasets import TIMIT
from sklearn.cluster import MiniBatchKMeans
import matplotlib.pyplot as plt
from tqdm import tqdm

# -----------------------------
# Config (tune these for your machine)
# -----------------------------
ROOT = os.environ.get("TIMIT_ROOT", "./timit_data")
SAMPLE_SUBSET = 800        # number of utterances to use (reduce for low-memory GPUs)
N_CLUSTERS = 60            # number of discrete observation symbols
N_STATES = 39              # phoneme states (reduced set)
BATCH_SIZE = 32
N_EM_ITERS = 8
SEED = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = "./checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

torch.manual_seed(SEED)
random.seed(SEED)

# -----------------------------
# Utilities: pad, batching
# -----------------------------

def pad_sequences(seqs: List[torch.Tensor], pad_value: int = -1):
    lengths = torch.tensor([s.size(0) for s in seqs], dtype=torch.long)
    maxT = int(lengths.max().item()) if len(lengths) > 0 else 0
    batch = seqs[0].new_full((len(seqs), maxT), pad_value)
    for i, s in enumerate(seqs):
        batch[i, : s.size(0)] = s
    return batch, lengths


def batch_generator(O_seqs: List[torch.Tensor], batch_size: int):
    for i in range(0, len(O_seqs), batch_size):
        batch = O_seqs[i : i + batch_size]
        padded, lengths = pad_sequences(batch, pad_value=-1)
        yield padded, lengths

# -----------------------------
# Feature extraction & discretization
# -----------------------------

def extract_mfccs_from_timit(root: str, subset: int = 1000, n_mfcc: int = 13):
    dataset = TIMIT(root=root, download=True, split='train')
    mfcc_transform = torchaudio.transforms.MFCC(sample_rate=16000, n_mfcc=n_mfcc, log_mels=True)

    features = []  # list of (T, n_mfcc) tensors
    utt_ids = []
    for i, data in enumerate(tqdm(dataset, desc="Loading TIMIT subset")):
        if i >= subset:
            break
        waveform, sample_rate, transcript, phoneme_seq, speaker_id, utterance_id = data
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        mfcc = mfcc_transform(waveform).squeeze(0).T  # (T_frames, n_mfcc)
        features.append(mfcc)
        utt_ids.append((speaker_id, utterance_id))
    return features, utt_ids


def build_kmeans(features: List[torch.Tensor], n_clusters: int = 60, batch_size: int = 1024):
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=SEED, batch_size=batch_size)
    # partial fit in streaming fashion
    for _ in range(2):
        for f in tqdm(features, desc="KMeans pass"):
            X = f.numpy()
            kmeans.partial_fit(X)
    return kmeans


def discretize_features(features: List[torch.Tensor], kmeans) -> List[torch.Tensor]:
    seqs = []
    for f in features:
        labels = kmeans.predict(f.numpy())
        seqs.append(torch.tensor(labels, dtype=torch.long))
    return seqs

# -----------------------------
# Log-space batched DP functions
# -----------------------------

NEG_INF = -1e9

def log_normalize(logp: torch.Tensor, dim: int) -> torch.Tensor:
    a = torch.logsumexp(logp, dim=dim, keepdim=True)
    return logp - a


def forward_log_batch(obs_batch: torch.LongTensor,
                      lengths: torch.LongTensor,
                      logA: torch.Tensor,
                      logB: torch.Tensor,
                      logpi: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    Bsz, T = obs_batch.shape
    N = logA.shape[0]

    pad_mask = (obs_batch >= 0)
    obs_clamped = obs_batch.clone()
    obs_clamped[~pad_mask] = 0

    logB_T = logB.t().contiguous()  # (M, N)
    emissions = logB_T[obs_clamped]  # (B, T, N)
    emissions = emissions.masked_fill(~pad_mask.unsqueeze(-1), NEG_INF)

    log_alpha = torch.full((Bsz, T, N), NEG_INF, device=logA.device, dtype=logA.dtype)
    log_alpha[:, 0, :] = logpi.unsqueeze(0) + emissions[:, 0, :]

    for t in range(1, T):
        prev = log_alpha[:, t - 1, :].unsqueeze(2)  # (B, N, 1)
        scores = prev + logA.unsqueeze(0)  # (B, N, N)
        trans_sum = torch.logsumexp(scores, dim=1)  # (B, N)
        log_alpha[:, t, :] = trans_sum + emissions[:, t, :]
        log_alpha[:, t, :] = log_alpha[:, t, :].masked_fill(~pad_mask[:, t].unsqueeze(-1), NEG_INF)

    last_idxs = (lengths - 1).clamp(min=0)
    final = log_alpha[torch.arange(Bsz, device=log_alpha.device), last_idxs, :]
    log_likelihoods = torch.logsumexp(final, dim=1)
    return log_likelihoods, log_alpha


def backward_log_batch(obs_batch: torch.LongTensor,
                       lengths: torch.LongTensor,
                       logA: torch.Tensor,
                       logB: torch.Tensor) -> torch.Tensor:
    Bsz, T = obs_batch.shape
    N = logA.shape[0]

    pad_mask = (obs_batch >= 0)
    obs_clamped = obs_batch.clone()
    obs_clamped[~pad_mask] = 0
    logB_T = logB.t().contiguous()
    emissions = logB_T[obs_clamped]
    emissions = emissions.masked_fill(~pad_mask.unsqueeze(-1), NEG_INF)

    log_beta = torch.full((Bsz, T, N), NEG_INF, device=logA.device, dtype=logA.dtype)
    for b in range(Bsz):
        last = lengths[b].item() - 1
        if last >= 0:
            log_beta[b, last, :] = 0.0

    for t in range(T - 2, -1, -1):
        term = logA.unsqueeze(0) + (emissions[:, t + 1, :].unsqueeze(1) + log_beta[:, t + 1, :].unsqueeze(1))
        log_beta[:, t, :] = torch.logsumexp(term, dim=2)
        log_beta[:, t, :] = log_beta[:, t, :].masked_fill(~pad_mask[:, t].unsqueeze(-1), NEG_INF)

    return log_beta


def viterbi_log_batch(obs_batch: torch.LongTensor,
                      lengths: torch.LongTensor,
                      logA: torch.Tensor,
                      logB: torch.Tensor,
                      logpi: torch.Tensor) -> List[torch.LongTensor]:
    Bsz, T = obs_batch.shape
    N = logA.shape[0]

    pad_mask = (obs_batch >= 0)
    obs_clamped = obs_batch.clone()
    obs_clamped[~pad_mask] = 0
    logB_T = logB.t().contiguous()
    emissions = logB_T[obs_clamped]
    emissions = emissions.masked_fill(~pad_mask.unsqueeze(-1), NEG_INF)

    delta = torch.full((Bsz, T, N), NEG_INF, device=logA.device, dtype=logA.dtype)
    psi = torch.zeros((Bsz, T, N), dtype=torch.long, device=logA.device)

    delta[:, 0, :] = logpi.unsqueeze(0) + emissions[:, 0, :]

    for t in range(1, T):
        scores = delta[:, t - 1, :].unsqueeze(2) + logA.unsqueeze(0)
        max_scores, argmax_states = scores.max(dim=1)
        delta[:, t, :] = max_scores + emissions[:, t, :]
        psi[:, t, :] = argmax_states
        delta[:, t, :] = delta[:, t, :].masked_fill(~pad_mask[:, t].unsqueeze(-1), NEG_INF)

    paths = []
    for b in range(Bsz):
        L = lengths[b].item()
        if L == 0:
            paths.append(torch.empty(0, dtype=torch.long))
            continue
        path = torch.empty(L, dtype=torch.long)
        last = L - 1
        path[last] = delta[b, last].argmax()
        for t in range(last, 0, -1):
            path[t - 1] = psi[b, t, path[t]]
        paths.append(path)
    return paths

# -----------------------------
# Fully vectorized B_num accumulation for Baum-Welch
# -----------------------------
def accumulate_B_counts!(B_num: torch.Tensor, gamma_flat: torch.Tensor, obs_flat: torch.Tensor):
    """
    Accumulate emission expected counts into B_num using flattened indexing.
    gamma_flat: (S, N) float tensor of posterior occupancies for valid frames
    obs_flat: (S,) long tensor of symbols in [0, M-1]
    B_num: (N, M) tensor to be updated in-place
    """
    # S: number of valid frames in the batch
    S, N = gamma_flat.shape
    if S == 0:
        return
    device = gamma_flat.device
    # idx_state repeated for each sample s: (S*N,)
    idx_state = torch.arange(N, device=device).unsqueeze(0).repeat(S, 1).reshape(-1)
    # idx_symbol: obs_flat repeated for each state -> (S*N,)
    idx_symbol = obs_flat.unsqueeze(1).repeat(1, N).reshape(-1)
    # linear indices into B_num.view(-1) of size N*M
    M = B_num.shape[1]
    lin_idx = idx_state * M + idx_symbol
    vals = gamma_flat.reshape(-1)
    # accumulate
    B_num.view(-1).index_add_(0, lin_idx, vals)

# Note: `!` in function name to indicate in-place update (not Pythonic but clear here)

# -----------------------------
# Batched Baum-Welch (EM) with vectorized B_num
# -----------------------------

def baum_welch_batch_optimized(O_seqs: List[torch.Tensor],
                     N: int,
                     M: int,
                     n_iter: int = 5,
                     batch_size: int = 16,
                     device: torch.device = DEVICE,
                     verbose: bool = True,
                     val_seqs: List[torch.Tensor] = None):
    torch.manual_seed(SEED)
    A = torch.rand(N, N, device=device)
    A = A / A.sum(dim=1, keepdim=True)
    B = torch.rand(N, M, device=device)
    B = B / B.sum(dim=1, keepdim=True)
    pi = torch.rand(N, device=device)
    pi = pi / pi.sum()

    logA = torch.log(A)
    logB = torch.log(B)
    logpi = torch.log(pi)

    history = {"train_ll": [], "val_ll": []}

    for iteration in range(n_iter):
        if verbose:
            print(f"EM iter {iteration+1}/{n_iter} ...")

        A_num = torch.zeros_like(A)
        B_num = torch.zeros_like(B)
        pi_num = torch.zeros_like(pi)

        total_frames = 0

        for padded, lengths in batch_generator(O_seqs, batch_size):
            padded = padded.to(device)
            lengths = lengths.to(device)
            Bsz, T = padded.shape

            log_like, log_alpha = forward_log_batch(padded, lengths, logA, logB, logpi)
            log_beta = backward_log_batch(padded, lengths, logA, logB)

            pad_mask = (padded >= 0)
            obs_clamped = padded.clone()
            obs_clamped[~pad_mask] = 0

            # emissions
            logB_T = logB.t()
            emissions = logB_T[obs_clamped]
            emissions = emissions.masked_fill(~pad_mask.unsqueeze(-1), NEG_INF)

            # gamma
            log_gamma = log_alpha + log_beta  # (B,T,N)
            log_gamma = log_gamma - log_like.view(-1, 1, 1)
            gamma = torch.exp(log_gamma).masked_fill(~pad_mask.unsqueeze(-1), 0.0)  # (B,T,N)

            # accumulate pi
            pi_num += gamma[:, 0, :].sum(dim=0)

            # vectorized B_num accumulation
            flat_mask = pad_mask.view(-1)
            if flat_mask.any():
                gamma_flat = gamma.view(-1, N)[flat_mask]  # (S, N)
                obs_flat = obs_clamped.view(-1)[flat_mask]  # (S,)
                accumulate_B_counts!(B_num, gamma_flat, obs_flat)
                total_frames += gamma_flat.shape[0]

            # xi accumulation optimized across batch/time
            # We'll compute xi_sum by iterating small T dimension (still faster than looping sequences)
            xi_sum = torch.zeros_like(A)
            for t in range(T - 1):
                mask_t = pad_mask[:, t] & pad_mask[:, t + 1]
                if not mask_t.any():
                    continue
                la_t = log_alpha[mask_t, t, :]  # (b_t, N)
                lb_t1 = log_beta[mask_t, t + 1, :]  # (b_t, N)
                obs_t1 = padded[mask_t, t + 1]  # (b_t,)
                emis_t1 = logB.t()[obs_t1]  # (b_t, N)
                term = la_t.unsqueeze(2) + logA.unsqueeze(0) + emis_t1.unsqueeze(1) + lb_t1.unsqueeze(1)
                log_xi_t = term - log_like[mask_t].view(-1, 1, 1)
                xi_sum += torch.exp(log_xi_t).sum(dim=0)

            A_num += xi_sum

        # M-step
        eps = 1e-12
        pi = (pi_num + eps) / ((pi_num.sum()) + eps)
        A = (A_num + eps) / (A_num.sum(dim=1, keepdim=True) + eps)
        B = (B_num + eps) / (B_num.sum(dim=1, keepdim=True) + eps)

        logA = torch.log(A)
        logB = torch.log(B)
        logpi = torch.log(pi)

        # compute training avg log-likelihood on a small random subset for tracking
        # sample up to 256 sequences
        sample_for_eval = random.sample(O_seqs, min(len(O_seqs), 256))
        padded_eval, lengths_eval = pad_sequences(sample_for_eval, pad_value=-1)
        padded_eval = padded_eval.to(device)
        lengths_eval = lengths_eval.to(device)
        train_ll_batch, _ = forward_log_batch(padded_eval, lengths_eval, logA, logB, logpi)
        avg_train_ll = train_ll_batch.mean().item()
        history['train_ll'].append(avg_train_ll)

        # validation
        if val_seqs is not None and len(val_seqs) > 0:
            sample_val = random.sample(val_seqs, min(len(val_seqs), 256))
            padded_val, lengths_val = pad_sequences(sample_val, pad_value=-1)
            padded_val = padded_val.to(device)
            lengths_val = lengths_val.to(device)
            val_ll_batch, _ = forward_log_batch(padded_val, lengths_val, logA, logB, logpi)
            avg_val_ll = val_ll_batch.mean().item()
            history['val_ll'].append(avg_val_ll)
        else:
            history['val_ll'].append(None)

        if verbose:
            print(f" Iter {iteration+1}: avg train LL = {avg_train_ll:.3f}, val LL = {history['val_ll'][-1]}")

        # checkpoint
        ckpt = {
            'A': A.cpu().numpy().tolist(),
            'B': B.cpu().numpy().tolist(),
            'pi': pi.cpu().numpy().tolist(),
            'history': history,
            'iteration': iteration,
        }
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"hmm_ckpt_iter{iteration+1}.json")
        with open(ckpt_path, 'w') as f:
            json.dump(ckpt, f)

    return A.cpu(), B.cpu(), pi.cpu(), history

# -----------------------------
# Plotting helper
# -----------------------------

def plot_history(history, save_path=None):
    plt.figure(figsize=(8,4))
    plt.plot(history['train_ll'], label='train avg LL')
    if history.get('val_ll') and any([v is not None for v in history['val_ll']]):
        plt.plot([v if v is not None else float('nan') for v in history['val_ll']], label='val avg LL')
    plt.xlabel('EM iteration')
    plt.ylabel('avg log-likelihood')
    plt.legend()
    plt.grid(True)
    if save_path:
        plt.savefig(save_path)
    plt.show()

# -----------------------------
# End-to-end training routine that runs extraction, discretization, training and plotting
# -----------------------------

def run_end_to_end():
    print('1) Extracting MFCCs...')
    features, utt_ids = extract_mfccs_from_timit(ROOT, subset=SAMPLE_SUBSET)

    print('2) Fitting MiniBatchKMeans...')
    kmeans = build_kmeans(features, n_clusters=N_CLUSTERS, batch_size=1024)

    print('3) Discretizing...')
    O_seqs = discretize_features(features, kmeans)

    random.shuffle(O_seqs)
    n_train = int(0.9 * len(O_seqs))
    train_seqs = O_seqs[:n_train]
    val_seqs = O_seqs[n_train:]

    print(f'Train sequences: {len(train_seqs)}, Val sequences: {len(val_seqs)}')

    print('4) Running batched Baum-Welch (EM) optimized on GPU...')
    A_tr, B_tr, pi_tr, history = baum_welch_batch_optimized(train_seqs, N_STATES, N_CLUSTERS,
                                                           n_iter=N_EM_ITERS, batch_size=BATCH_SIZE,
                                                           device=DEVICE, verbose=True, val_seqs=val_seqs)

    print('5) Plotting training history...')
    plot_history(history, save_path=os.path.join(CHECKPOINT_DIR, 'train_history.png'))

    print('6) Viterbi decode a few validation sequences...')
    padded, lengths = pad_sequences(val_seqs[:BATCH_SIZE], pad_value=-1)
    paths = viterbi_log_batch(padded.to(DEVICE), lengths.to(DEVICE), torch.log(A_tr.to(DEVICE)), torch.log(B_tr.to(DEVICE)), torch.log(pi_tr.to(DEVICE)))
    print(f'Decoded {len(paths)} validation sequences.')

    # Save final model
    final = {'A': A_tr.numpy().tolist(), 'B': B_tr.numpy().tolist(), 'pi': pi_tr.numpy().tolist(), 'history': history}
    with open(os.path.join(CHECKPOINT_DIR, 'hmm_final.json'), 'w') as f:
        json.dump(final, f)

    return A_tr, B_tr, pi_tr, history

# If running as a script
if __name__ == '__main__':
    A_tr, B_tr, pi_tr, history = run_end_to_end()
    print('Done')
